# deckard Layers CLI Tour

This notebook is the final bridge between the tutorial notebooks and using the Deckard CLI directly.

It executes real `python -m deckard ...` commands in the `examples/sklearn` context and reuses the study-backed outputs created by `optimize.ipynb`. That keeps the command examples tied to the same config tree and runtime conventions used in the sklearn example project.

The command surface covered here comes from `deckard/layers/__init__.py`:

- `optimize`
- `compile_results`
- `find_best`
- `progress_bar`
- `plot`
- `declarations`
- `plugins`
- `frameworks`
- `rerun_failed_studies`
- `survival` when optional survival dependencies are installed

Use this notebook alongside the more execution-heavy notebooks:

- `hydra.ipynb` for composition and override syntax
- `optimize.ipynb` for single-run and multirun optimization behavior
- `optuna.ipynb` for study, storage, sampler, and pruner behavior
- `artifacts.ipynb` for persisted outputs and artifact handling

In [1]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

import pandas as pd

from deckard.layers import SUPPORTED_LAYERS

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "examples").exists():
    PROJECT_ROOT = Path("../..").resolve()

EXAMPLES_DIR = PROJECT_ROOT / "examples" / "sklearn"
CONFIG_DIR = EXAMPLES_DIR / "config"
CONFIG_DIR_STR = str(CONFIG_DIR)
DOCS_NOTEBOOK_DIR = PROJECT_ROOT / "docs" / "notebooks"
BUILD_DIR = DOCS_NOTEBOOK_DIR / "build" / "deckard_layers"
BUILD_DIR.mkdir(parents=True, exist_ok=True)

OPTIMIZE_BUILD_DIR = DOCS_NOTEBOOK_DIR / "build" / "optimize_notebook"
OPTUNA_DB = OPTIMIZE_BUILD_DIR / "multirun" / "optuna_phase6.db"
OPTUNA_STORAGE = f"sqlite:///{OPTUNA_DB.as_posix()}"
DVC_FILE = EXAMPLES_DIR / "dvc.yaml"
META_SCHEMA = CONFIG_DIR / "meta.yaml"
SURVIVAL_CFG = CONFIG_DIR / "survival.yaml"

DECKARD_PY_CMD = [sys.executable, "-m", "deckard"]

env = os.environ.copy()
env["DECKARD_CONFIG_DIR"] = CONFIG_DIR.as_posix()
env["DECKARD_DEFAULT_CONFIG_FILE"] = "default.yaml"
env.setdefault("DECKARD_TEST_MAX_SAMPLES", "64")
env.setdefault("TQDM_DISABLE", "1")


def run_cli(
    args: list[str],
    *,
    cwd: Path = EXAMPLES_DIR,
    check: bool = True,
    timeout: float | None = None,
    ) -> subprocess.CompletedProcess[str]:
    print("$", " ".join(args))
    result = subprocess.run(
        args,
        cwd=cwd.as_posix(),
        env=env,
        capture_output=True,
        text=True,
        timeout=timeout,
        check=False,
    )
    if result.stdout.strip():
        print(result.stdout)
    if result.stderr.strip():
        print(result.stderr)
    if check and result.returncode != 0:
        raise RuntimeError(
            f"Command failed with exit code {result.returncode}: {' '.join(args)}",
        )
    return result


def run_cli_timeout(
    args: list[str],
    *,
    cwd: Path = EXAMPLES_DIR,
    timeout: float = 2.0,
    ) -> dict[str, str | bool | int]:
    print("$", " ".join(args))
    try:
        result = subprocess.run(
            args,
            cwd=cwd.as_posix(),
            env=env,
            capture_output=True,
            text=True,
            timeout=timeout,
            check=False,
        )
        stdout = result.stdout
        stderr = result.stderr
        returncode = result.returncode
        timed_out = False
    except subprocess.TimeoutExpired as exc:
        stdout = exc.stdout or ""
        stderr = exc.stderr or ""
        returncode = -1
        timed_out = True
        print("Command timed out after previewing live output.")

    if stdout.strip():
        print(stdout)
    if stderr.strip():
        print(stderr)

    return {
        "timed_out": timed_out,
        "returncode": returncode,
        "stdout": stdout,
        "stderr": stderr,
    }


required_paths = [
    CONFIG_DIR / "default.yaml",
    DVC_FILE,
    META_SCHEMA,
    OPTUNA_DB,
    SURVIVAL_CFG,
]
for path in required_paths:
    assert path.exists(), f"Required path is missing: {path}"

print(f"PROJECT_ROOT={PROJECT_ROOT}")
print(f"EXAMPLES_DIR={EXAMPLES_DIR}")
print(f"CONFIG_DIR={CONFIG_DIR}")
print(f"BUILD_DIR={BUILD_DIR}")
print(f"OPTUNA_DB={OPTUNA_DB}")
print(f"SUPPORTED_LAYERS={SUPPORTED_LAYERS}")

/Users/c.meyers/.pyenv/versions/deckard/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PROJECT_ROOT=/Users/c.meyers/Documents/deckard
EXAMPLES_DIR=/Users/c.meyers/Documents/deckard/examples/sklearn
CONFIG_DIR=/Users/c.meyers/Documents/deckard/examples/sklearn/config
BUILD_DIR=/Users/c.meyers/Documents/deckard/docs/notebooks/build/deckard_layers
OPTUNA_DB=/Users/c.meyers/Documents/deckard/docs/notebooks/build/optimize_notebook/multirun/optuna_phase6.db
SUPPORTED_LAYERS=['compile_results', 'declarations', 'progress_bar', 'plot', 'optimize', 'plugins', 'frameworks', 'find_best', 'rerun_failed_studies', 'survival']


## 1) Shared CLI Setup in `examples/sklearn`

All commands in this notebook stay anchored to the existing config tree under `examples/sklearn/config`. That keeps the CLI examples aligned with the same defaults, aliases, and study naming conventions used by the example project and the other tutorial notebooks.

The shared assumptions are:

- commands run from `examples/sklearn`
- `DECKARD_CONFIG_DIR=./config` and `DECKARD_DEFAULT_CONFIG_FILE=default.yaml` are set in the environment
- `optimize.ipynb` has already produced a multirun study we can reuse for downstream CLI layers
- when a command needs a profile, it uses `default.yaml` or `survival.yaml` from `examples/sklearn/config`

The sections below execute the CLI directly so you can see the exact arguments that bridge the tutorial flow into command-line usage.

## 2) Core Runtime Layers

These are the runtime-oriented layer commands that most directly connect the `examples/sklearn` configs to study execution and post-run analysis.

This section does four things with real CLI commands:

- runs a tiny single-run `optimize` smoke command against `default.yaml`
- compiles the multirun study from `optimize.ipynb` into a tabular results file with `compile_results`
- exports a runnable best-trial config with `find_best`
- previews `progress_bar` as a live monitoring command against the sklearn DVC stage plan

In [20]:
!cd ../../examples/sklearn && deckard compile_results --output_file results.csv --optuna_db sqlite:///optuna.db

/Users/c.meyers/.pyenv/versions/deckard/bin/deckard:8: UserWarning: 
'score/classification' is validated against ConfigStore schema with the same name.
This behavior is deprecated in Hydra 1.1 and will be removed in Hydra 1.2.
See https://hydra.cc/docs/1.2/upgrades/1.0_to_1.1/automatic_schema_matching for migration instructions.
  sys.exit(main())
/Users/c.meyers/.pyenv/versions/deckard/lib/python3.10/site-packages/hydra/main.py:94: UserWarning: 
'score/classification' is validated against ConfigStore schema with the same name.
This behavior is deprecated in Hydra 1.1 and will be removed in Hydra 1.2.
See https://hydra.cc/docs/1.2/upgrades/1.0_to_1.1/automatic_schema_matching for migration instructions.
  _run_hydra(
/Users/c.meyers/.pyenv/versions/deckard/lib/python3.10/site-packages/hydra/_internal/callbacks.py:28: UserWarning: Callback DefaultOptimizerCallback.on_run_start raised InterpolationToMissingValueError: MissingMandatoryValue while resolving interpolation: Missing mandatory

In [22]:
results_df = pd.read_csv("../../examples/sklearn/results.csv")
results_df

,number,0,1,2,datetime_start,datetime_complete,duration,++attack.attack_params.delta,++attack.attack_params.epsilon,++attack.attack_params.max_iter,...,state,study_name,++defense.defense_params.max_value,++model.model_params.C,++model.model_params.max_iter,++model.model_params.penalty,++model.model_params.solver,++model.model_params.tol,++defense.defense_params.cutoff,++defense.defense_params.scale
